<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 05 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Explore Doris Functions and Build Analytical Queries</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:920px;margin:0">Choose a function by the shape of the result it must produce, then turn event-detail rows into grouped metrics, filtered groups, daily trends, and a guided business result.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Scalar · Aggregate · TVF · Window · CTE · Lambda · UDF boundary</span>
</div>

This lab continues with the complete `events_modelled` table created in Module 4. The notebook supplies runnable SQL and small interactions so that you can focus on what each function does to its input rows and output rows.


### Initialize the Lab

Run the next cell before Section 1. It loads the shared course helper and creates the `lab` object used by every later cell. Run it again after restarting the Jupyter kernel. It does not start Docker or change data in Doris.


In [16]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


## 1. Choose a function category from the required result shape

Doris functions answer different kinds of questions:

| Function category | Input and output behavior | Example |
|---|---|---|
| Scalar function | Produces one value for each input row | `LOWER(event_type)` |
| Aggregate function | Combines values from multiple rows into one value per group | `SUM(revenue)` |
| Table-valued function (TVF) | Produces a relation that can appear in `FROM` | `S3(...)` |
| Window function | Adds a calculated value while retaining the input result rows | `ROW_NUMBER() OVER (...)` |

Match each expression to the requirement it satisfies. Use the row-shape behavior—not only the function name—to decide.


In [2]:
lab.function_category_activity();


<IPython.core.display.Javascript object>

## 2. Turn stored values into reporting dimensions

Connect to the Frontend (FE), select the course database, and inspect eight matching events. The table keeps the original `event_time` and `region`; scalar functions derive reporting values when the query runs:

- `TO_DATE` converts a timestamp to its calendar date.
- `DATE_TRUNC(..., 'hour')` normalizes timestamps to an hourly boundary.
- `EXTRACT` returns one time component.
- `REPLACE` and `UPPER` produce a presentation label.
- `LIKE` and `REGEXP` select rows by a string pattern.

Each selected input event still produces one output row, so these scalar functions do not change the result grain.


In [9]:
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

lab.sql("""
SELECT
    event_time,
    TO_DATE(event_time) AS event_date,
    DATE_TRUNC(event_time, 'hour') AS event_hour,
    EXTRACT(HOUR FROM event_time) AS hour_of_day,
    event_type,
    UPPER(REPLACE(region, 'region_', 'REGION-')) AS region_label
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
  AND region LIKE 'region_0%'
  AND event_type REGEXP '^(view|purchase)$'
ORDER BY event_time, event_id
LIMIT 8
""", title="Scalar-function result");


event_time,event_date,event_hour,hour_of_day,event_type,region_label
2020-03-03 00:00:16,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-01
2020-03-03 00:00:24,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-07
2020-03-03 00:00:29,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-05
2020-03-03 00:00:54,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-03
2020-03-03 00:01:18,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-01
2020-03-03 00:01:24,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-01
2020-03-03 00:01:30,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-08
2020-03-03 00:01:56,2020-03-03,2020-03-03 00:00:00,0,purchase,REGION-02


**Expected result:** eight rows are returned. The first row has `event_date = 2020-03-03`, `event_hour = 2020-03-03 00:00:00`, `hour_of_day = 0`, and `region_label = REGION-01`. Notice that the stored values remain available beside the derived values.


## 3. Reduce detail rows to grouped business metrics

The next query changes the grain from one row per original event to one row per date and region. `COUNT`, `MIN`, `MAX`, and `SUM` are aggregate functions because they combine values across many input rows. For the selected day, 140,782 event rows become eight grouped rows.

The purchase calculations place a conditional expression inside each aggregate function. Non-purchase rows still contribute to `event_count`, but they contribute `NULL` to the purchase `MIN` and `MAX`, and zero to `purchase_revenue`. This produces general activity and purchase-specific measures in the same grouped row.


In [10]:
lab.sql("""
SELECT
    TO_DATE(event_time) AS event_date,
    region,
    COUNT(*) AS event_count,
    MIN(CASE WHEN event_type = 'purchase' THEN revenue END) AS min_purchase_value,
    MAX(CASE WHEN event_type = 'purchase' THEN revenue END) AS max_purchase_value,
    SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS purchase_revenue
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY TO_DATE(event_time), region
ORDER BY purchase_revenue DESC, region
""", title="Daily metrics by region");


event_date,region,event_count,min_purchase_value,max_purchase_value,purchase_revenue
2020-03-03,region_02,17838,1.80,2316.64,814438.41
2020-03-03,region_08,17388,1.85,2038.66,803421.62
2020-03-03,region_06,17929,2.57,2205.95,789599.79
2020-03-03,region_01,17724,0.97,2573.81,786295.23
2020-03-03,region_03,17634,2.42,2510.68,784931.15
2020-03-03,region_07,17407,1.03,2110.48,777607.63
2020-03-03,region_04,17175,1.00,2574.04,763191.11
2020-03-03,region_05,17687,2.26,2568.92,727071.45


**Expected result:** eight grouped rows are returned—one for each region on `2020-03-03`. Adding the eight `event_count` values gives **140,782** input events, while adding `purchase_revenue` gives **6,246,556.39**. `min_purchase_value` and `max_purchase_value` describe only purchase rows because aggregate functions ignore the `NULL` produced for other event types.


## 4. Filter input rows with WHERE and aggregated groups with HAVING

This query asks which products generated at least 120,000 in purchase revenue on one day.

| Clause | Operates on | Meaning in this query |
|---|---|---|
| `WHERE` | Original event rows | Only purchase events from the selected day enter aggregation |
| `GROUP BY` | Filtered input rows | One group is created for each `product_id` |
| `HAVING` | Aggregated groups | Only product groups with at least 120,000 in revenue remain |

Moving the revenue threshold into `WHERE` would ask a different question: it would filter individual purchases rather than product totals.


In [ ]:
lab.sql("""
SELECT
    product_id,
    COUNT(*) AS purchase_events,
    SUM(revenue) AS purchase_revenue
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
  AND event_type = 'purchase'
GROUP BY product_id
HAVING SUM(revenue) >= 120000
ORDER BY purchase_revenue DESC, product_id
""", title="Product groups above the revenue threshold");


**Expected result:** six product groups remain after `HAVING`. Product `1005115` ranks first with 593 purchase events and 516,793.33 in revenue; product `1004249` is the last qualifying group with 126,135.40 in revenue.


## 5. Use ANY_VALUE only when any representative value is valid

A grouped query may project a column only when that column defines the group or is calculated by an aggregate function. The first query deliberately violates that rule: it groups by `region` but also requests an unaggregated `event_time` value. Doris rejects the ambiguous projection instead of choosing a timestamp silently.


In [11]:
lab.expected_sql_error("""
SELECT
    region,
    TO_DATE(event_time) AS event_date,
    COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY region
ORDER BY region
""", contains="not in aggregate", title="Doris rejects an ambiguous grouped projection");


The date predicate guarantees that every selected event belongs to the same calendar date. Under that specific condition, choosing any non-`NULL` date from each region is meaningful, so `ANY_VALUE` makes the intention explicit. Do not use it to hide a dimension that should define separate groups.


In [12]:
lab.sql("""
SELECT
    region,
    ANY_VALUE(TO_DATE(event_time)) AS event_date,
    COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-03 00:00:00'
  AND event_time <  '2020-03-04 00:00:00'
GROUP BY region
ORDER BY region
""", title="A valid representative date for each region");


region,event_date,event_count
region_01,2020-03-03,17724
region_02,2020-03-03,17838
region_03,2020-03-03,17634
region_04,2020-03-03,17175
region_05,2020-03-03,17687
region_06,2020-03-03,17929
region_07,2020-03-03,17407
region_08,2020-03-03,17388


**Expected result:** eight region rows are returned and every `event_date` is `2020-03-03`. Their `event_count` values still sum to 140,782. `ANY_VALUE` is valid here because the date predicate makes the representative date identical across the possible input values.


## 6. Name an aggregated intermediate result with a CTE

A Common Table Expression (CTE) gives a name to an intermediate result within one statement. Here, `daily` names the result produced after event-detail rows have been aggregated to one row per date. It does not create a permanent table.

This first query deliberately stops after the aggregation so that the eight-row input to the next section is visible.


In [13]:
lab.sql("""
WITH daily AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS daily_revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-09 00:00:00'
    GROUP BY TO_DATE(event_time)
)
SELECT
    event_date,
    daily_revenue
FROM daily
ORDER BY event_date
""", title="Daily aggregate produced by a CTE");


event_date,daily_revenue
2020-03-01,5670241.29
2020-03-02,9933097.98
2020-03-03,6246556.39
2020-03-04,0.00
2020-03-05,0.00
2020-03-06,0.00
2020-03-07,0.00
2020-03-08,0.00


**Expected result:** aggregation produces eight daily rows. Revenue is 5,670,241.29 on `2020-03-01`, 9,933,097.98 on `2020-03-02`, and 6,246,556.39 on `2020-03-03`; the remaining selected dates have zero purchase revenue. These eight rows become the input result rows for the window functions in the next section.


## 7. Add window calculations while retaining the daily rows

The query begins with the same `daily` CTE and therefore the same eight daily rows. Window functions then add calculations to each row:

- `LAG` reads the previous daily result.
- `SUM(...) OVER (...)` produces a running total.
- `ROW_NUMBER` assigns a deterministic revenue rank.

Unlike the earlier aggregate functions, these window functions do not collapse the eight input result rows.


In [14]:
lab.sql("""
WITH daily AS (
    SELECT
        TO_DATE(event_time) AS event_date,
        SUM(CASE WHEN event_type = 'purchase' THEN revenue ELSE 0 END) AS daily_revenue
    FROM events_modelled
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-09 00:00:00'
    GROUP BY TO_DATE(event_time)
)
SELECT
    event_date,
    daily_revenue,
    LAG(daily_revenue, 1, 0) OVER (ORDER BY event_date) AS previous_revenue,
    SUM(daily_revenue) OVER (
        ORDER BY event_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue,
    ROW_NUMBER() OVER (
        ORDER BY daily_revenue DESC, event_date
    ) AS revenue_rank
FROM daily
ORDER BY event_date
""", title="Window calculations over the daily rows");


event_date,daily_revenue,previous_revenue,cumulative_revenue,revenue_rank
2020-03-01,5670241.29,0.00,5670241.29,3
2020-03-02,9933097.98,5670241.29,15603339.27,1
2020-03-03,6246556.39,9933097.98,21849895.66,2
2020-03-04,0.00,6246556.39,21849895.66,4
2020-03-05,0.00,0.00,21849895.66,5
2020-03-06,0.00,0.00,21849895.66,6
2020-03-07,0.00,0.00,21849895.66,7
2020-03-08,0.00,0.00,21849895.66,8


**Expected result:** the output still contains the same eight dates produced by the `daily` CTE, but each row now has previous-revenue, cumulative-revenue, and rank values. On `2020-03-02`, the previous revenue is 5,670,241.29 and the cumulative revenue is 15,603,339.27.

| Query stage | Input rows | Output rows | Row behavior |
|---|---:|---:|---|
| Daily aggregation | Event-detail rows | 8 | Rows are collapsed to daily grain |
| Window calculation | 8 daily rows | 8 | Daily rows are retained and calculated columns are added |


## 8. Apply a Lambda expression to each ARRAY element

Doris Lambda expressions commonly appear inside higher-order ARRAY functions. `ARRAY_MAP` transforms each element, while `ARRAY_FILTER` keeps elements that satisfy a condition. This small literal isolates the behavior; it does not change the `events_modelled` schema.


In [ ]:
lab.sql("""
SELECT
    ARRAY_MAP(x -> x * 2, [1, 2, 3, 4]) AS doubled,
    ARRAY_FILTER(x -> x >= 3, [1, 2, 3, 4]) AS kept
""", title="Lambda expressions in ARRAY functions");


**Expected result:** `doubled` is `[2, 4, 6, 8]` and `kept` is `[3, 4]`. The Lambda expression is an argument to a built-in higher-order function; it is not a separately deployed user-defined function (UDF).


## 9. Assemble a business query from bounded choices

Choose a reporting grain, an event filter, and a metric. The notebook generates visible Doris SQL and runs it, so you can connect each choice to its expression without writing an entire statement from an empty editor.

Every selectable operation in this builder uses a Doris built-in function: `TO_DATE` and `DATE_TRUNC` define the reporting grain, while `COUNT`, `COUNT(DISTINCT ...)`, and `SUM` calculate metrics. These functions are already provided by Doris and require no external implementation or registration.

The default choices ask for daily purchase revenue. Change one choice at a time and observe which clause or expression changes and whether the output grain changes.


In [4]:
lab.guided_analysis_builder();


**Expected result:** with the default choices—Day, Purchases only, and Revenue—the leading result is `2020-03-02` with 9,933,097.98, followed by `2020-03-03` with 6,246,556.39 and `2020-03-01` with 5,670,241.29. Selecting Week or Month changes the grouping grain; selecting another metric changes the aggregate expression.

The generated query is composed entirely from built-in functions. The S3 function used in Module 3 is also built in; it belongs to the TVF category because it returns a temporary relation for `FROM`. This lab does not reread the remote dataset because the function-category behavior can be identified without repeating a full object-storage load.


## 10. Recognize when a reusable UDF is justified

A User-Defined Function (UDF) extends Doris with logic supplied by the user. It is appropriate when required domain logic cannot be expressed clearly with built-in functions and must be reused by many queries. For example, a company might own a risk-scoring algorithm that uses a private model and has no Doris built-in equivalent.

| Requirement | Appropriate choice |
|---|---|
| Convert a timestamp to a date | Built-in `TO_DATE` |
| Calculate grouped revenue | Built-in `SUM` |
| Rank rows within each region | Built-in `ROW_NUMBER` |
| Apply a short expression once in one query | A normal SQL expression |
| Apply an expression to each ARRAY element | Lambda with a built-in higher-order function |
| Reuse unsupported company-specific logic across queries | Consider a UDF |

A Java scalar UDF follows this lifecycle:

```text
Implement an evaluate method
        ↓
Package the implementation as a JAR
        ↓
Make the JAR available to Doris nodes
        ↓
Register its SQL signature with CREATE FUNCTION
        ↓
Call it from SELECT like another scalar function
```

The following example is a registration template, not a runnable cell. It requires a real JAR containing the named Java class and administrative privileges:

```sql
CREATE FUNCTION customer_risk_score(BIGINT, DECIMAL(12, 2))
RETURNS DOUBLE
PROPERTIES (
    "file" = "https://example.org/functions/customer-risk.jar",
    "symbol" = "com.example.analytics.CustomerRiskScore",
    "type" = "JAVA_UDF",
    "always_nullable" = "true",
    "volatility" = "immutable"
);

SELECT
    event_id,
    customer_risk_score(user_id, revenue) AS risk_score
FROM events_modelled;
```

`CREATE FUNCTION` does not contain the risk algorithm. It registers the SQL name, input and return types, implementation file, and implementation class. The Backend (BE) invokes that implementation when the query runs. Mark a function `immutable` only when the same inputs always produce the same output.

**Expected understanding:** all executable analysis in this lab should remain on built-in functions. A deployed UDF is an extension point for unavailable, reusable domain logic—not a replacement for `TO_DATE`, `SUM`, `ROW_NUMBER`, or a clear SQL expression.


### Stop the Doris sandbox

Run this optional cell to release CPU and memory. Lab 5 reads but does not replace `events_modelled`; Docker named volumes preserve the database and table.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");


**Expected result:** Docker reports `container=exited`.

### Restart the Doris sandbox

Run this cell before continuing to another module. It starts the existing container, reconnects to FE, and verifies the persisted query-ready model.


In [ ]:
lab.shell(r"""
set -euo pipefail

docker start doris
docker inspect --format 'container={{.State.Status}} health={{.State.Health.Status}}' doris
""", title="Start the Doris sandbox")

lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.sql("SELECT COUNT(*) AS event_rows FROM events_modelled", title="Recovered analytical input");


**Expected result:** the container returns to `healthy`, and `event_rows` is 10,158,080.

## Lab complete

You selected functions by their result shape, transformed individual values, reduced detail rows to grouped metrics, separated `WHERE` from `HAVING`, used `ANY_VALUE` only under a valid constraint, named an intermediate result with a CTE, observed window functions retain its rows, applied Lambda expressions to ARRAY values, assembled a guided analytical query from built-in functions, and identified when unsupported reusable logic may justify a UDF.

Official references: [SELECT](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/data-query/SELECT/) · [Common Table Expressions](https://doris.apache.org/docs/4.x/query-data/cte/) · [Window Functions](https://doris.apache.org/docs/4.x/query-data/window-function/) · [ANY_VALUE](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/aggregate-functions/any-value/) · [ARRAY_MAP](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/scalar-functions/array-functions/array-map/) · [S3 TVF](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/s3/) · [Java UDF, UDAF, UDWF and UDTF](https://doris.apache.org/docs/4.x/query-data/udf/java-user-defined-function/) · [CREATE FUNCTION](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/function/CREATE-FUNCTION/)
